In [68]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [69]:
load_dotenv()

True

In [70]:
llm=ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=1.5,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [71]:
class llmState(TypedDict):
    user_input: str
    llm_output: str

In [72]:
def llm_qa(state:llmState)->llmState:
    question=state['user_input']
    prompt=f'You are a helpful AI assistant, answer all the question in one line. \n {question}'
    output=llm.invoke(prompt).content
    state['llm_output']=output
    return state


In [73]:
graph = StateGraph(llmState)

graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

workflow=graph.compile()

In [74]:
initial_state={'user_input':'How far is moon from earth'}
final_state=workflow.invoke(initial_state)
print(final_state['llm_output'])

The Moon is approximately 238,855 miles or 384,400 kilometers away from Earth on average.


In [76]:
#### PROMPT CHAINING USING LANGGRAPH
class llmChainState(TypedDict):
    input:str
    output:str

def blog_gen(state:llmChainState) -> llmChainState:
    input=state['input']
    prompt=f'write a 3-4 line blog on {input}'
    output=llm.invoke(prompt).content
    state['output']=output
    return state

def caption_gen(state:llmChainState)->llmChainState:
    input=state['input']
    prompt=f'Write 4-5 hashtags for the blog \n {input}'
    output=llm.invoke(prompt).content
    state['output']=output
    return state    

graph=StateGraph(llmChainState)
graph.add_node('blog_node',blog_gen)
graph.add_node('caption_node',caption_gen)
graph.add_edge(START,'blog_node')
graph.add_edge('blog_node','caption_node')
graph.add_edge('caption_node',END)

workflow=graph.compile()

initial_state={'input':'Goa'}
final_state=workflow.invoke(initial_state)
print(final_state['output'])

Here are five well-thought-out hashtags that encapsulate the essence of Goa, each targeting a different aspect of its appeal:

1. #Goa
2. #BeachLife
3. #IbizaOfTheEast
4. #GoaCulture
5. #TravelDiaries
